In [7]:
import json

%load_ext autoreload
%autoreload 2

from utgen.raggraph.walker import build_graph_from_directory
from utgen.raggraph.utils import get_node_context
from utgen.test_generation_crew.crew import TestGenerationCrew
from utgen.validation import validate_individual_test, save_and_clean_tests

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
g = build_graph_from_directory(code_path="../src", save_graph=False)

In [5]:
tests_results: dict[str, dict[str, dict[str, str]]] = {}

# TODO: afegir guardrails que falten
test_generator = TestGenerationCrew(guardrail_max_retries=3, verbose=False)

# Per a cada node del graf, obtenim el seu context i creem el test
for node_id, data in list(g.nodes(data=True)):
    if data["type"] in ["method"]:
    # if data["type"] in ["function", "method"]:
        print(f"Generating tests for node: {node_id}")
        # Get context
        context = get_node_context(g=g, node_id=node_id)

        # Generate tests
        inputs = {"graph_context": context}
        response = test_generator.crew().kickoff(inputs=inputs)

        # Convert string to dictionary
        response_dict = json.loads(response.raw)

        # Store results
        k = data["file"].replace("/", "___").replace(".py", f"___{data['name']}")
        tests_results[k] = response_dict['tests']

Generating tests for node: utgen/raggraph/parser.py::method::CodeGraphBuilder1.__init__
Generating tests for node: utgen/raggraph/parser.py::method::CodeGraphBuilder1.visit_ClassDef
Generating tests for node: utgen/raggraph/parser.py::method::CodeGraphBuilder1.visit_FunctionDef
Generating tests for node: utgen/raggraph/parser.py::method::CodeGraphBuilder2.__init__
Generating tests for node: utgen/raggraph/parser.py::method::CodeGraphBuilder2.attach_parents
Generating tests for node: utgen/raggraph/parser.py::method::CodeGraphBuilder2.visit_ClassDef
Generating tests for node: utgen/raggraph/parser.py::method::CodeGraphBuilder2.visit_FunctionDef
Generating tests for node: utgen/raggraph/parser.py::method::CodeGraphBuilder2.visit_AsyncFunctionDef
Generating tests for node: utgen/raggraph/parser.py::method::CodeGraphBuilder2.is_local_symbol
Generating tests for node: utgen/raggraph/parser.py::method::CodeGraphBuilder2.visit_Name
Generating tests for node: utgen/raggraph/parser.py::method::

In [ ]:
for k1, v1 in tests_results.items():
    # Validem el test generat
    print(f"\nValidant tests per `{k1}`...")
    base_import = 'from ' + '.'.join(k1.split("___")[:-1]) + ' import ' + k1.split("___")[-1].split(".")[0]
    print(f"Base import afegit: {base_import}")
    filename = f"test_{k1.replace('.', '_')}.py"
    tests_que_han_passat = []

    for k2, v2 in v1.items():
        name, imp, code = k2, v2['imports'], v2['code']
        # Afegim el import base per si no està present
        if base_import not in imp:
            print(f"⚠️ Import base `{base_import}` no present en el test `{name}`, afegint-lo...")
            imp.append(base_import)
        imp = "\n".join(imp)
        if validate_individual_test(import_code=imp, test_code=code):
            print(f"✅ Test `{name}` acceptat")
            tests_que_han_passat.append((imp, code))
        else:
            print(f"❌ Test `{name}` rebutjat")
        print(imp)
        print(code)

    # Guardem i deixem el fitxer perfecte
    save_and_clean_tests(valid_tests=tests_que_han_passat, destination=f"../tests/{filename}")

# TODO: run pytest coverage


Validant tests per `utgen___raggraph___parser___CodeGraphBuilder1.__init__`...
Base import afegit: from utgen.raggraph.parser import CodeGraphBuilder1
❌ Test `test_init_with_valid_paths` rebutjat
import pytest
from unittest.mock import Mock
import os
import networkx as nx
from utgen.raggraph.parser import CodeGraphBuilder1
def test_init_with_valid_paths(tmp_path):
    """
    Test CodeGraphBuilder1.__init__ with valid code_path and file_path.
    """
    # Arrange
    code_path = str(tmp_path / "project")
    os.makedirs(code_path)
    
    file_path = code_path / "src" / "module.py"
    os.makedirs(file_path.parent)
    file_path.write_text("def example():", encoding="utf-8")
    
    graph = nx.DiGraph()
    
    # Act
    builder = CodeGraphBuilder1(code_path, str(file_path), graph)
    
    # Assert
    assert builder.file_path == str(file_path)
    assert builder.file_path_clean == "src/module.py"
    assert builder.graph is graph
    assert builder.current_file_id == "src::modul

In [ ]:
# TODO: script final que reordene la carpeta tests: un test.py per cada .py del src
